In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_finelabels_liana"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_finelabels_liana"
for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# ----------------------------
# Cell 2 — LIANA runner function, parameterised by groupby column so
# the same logic works for cell_type_fine. Same exclusion discipline,
# same min-cell threshold, same rank_aggregate settings as original.
# ----------------------------
EXCLUDE_FROM_LIANA = ["Unassigned (n=28, doublet/mixed-identity artefact)",
                       "Unassigned (n=91, stromal/RBC contamination artefact)"]
MIN_CELLS_FOR_LIANA = 50

def run_liana_fine(adata, subset_mask, label, groupby_col="cell_type_fine", min_cells=MIN_CELLS_FOR_LIANA):
    adata_sub = adata[subset_mask].copy()
    adata_sub = adata_sub[adata_sub.obs[groupby_col].notna()].copy()
    adata_sub = adata_sub[~adata_sub.obs[groupby_col].isin(EXCLUDE_FROM_LIANA)].copy()

    cell_counts = adata_sub.obs[groupby_col].value_counts()
    viable_types = cell_counts[cell_counts >= min_cells].index.tolist()
    excluded_types = cell_counts[cell_counts < min_cells].index.tolist()
    if excluded_types:
        print(f"  Excluding from {label} (< {min_cells} cells): {excluded_types}")

    adata_sub = adata_sub[adata_sub.obs[groupby_col].isin(viable_types)].copy()
    adata_sub.obs[groupby_col] = adata_sub.obs[groupby_col].astype(str)
    print(f"  Running LIANA on {label}: {adata_sub.n_obs} cells, {adata_sub.obs[groupby_col].nunique()} cell types")

    li.mt.rank_aggregate(
        adata_sub,
        groupby=groupby_col,
        expr_prop=0.1,
        verbose=False,
        use_raw=False,
    )
    results = adata_sub.uns["liana_res"].copy()
    del adata_sub
    gc.collect()
    return results

print("LIANA runner ready (fine-label version)")

LIANA runner ready (fine-label version)


In [3]:
# ----------------------------
# Cell 3 — Load both datasets with cell_type_fine, run LIANA per condition
# ----------------------------
adata1 = sc.read_h5ad(PROCESSED_DIR / "GSE114725_phase2_v2_annotated_finelabels.h5ad")
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected_finelabels.h5ad")

print(f"GSE114725: {adata1.n_obs} cells")
print(f"GSE176078: {adata2.n_obs} cells")

all_liana_results_fine = {}

# GSE114725 — Tumour and Normal (same two conditions as original)
for tissue in ["TUMOR", "NORMAL"]:
    mask = adata1.obs["tissue"] == tissue
    label = f"GSE114725_{tissue}"
    results = run_liana_fine(adata1, mask.values, label)
    all_liana_results_fine[label] = results
    results.to_csv(RESULTS_DIR / f"{label}_liana_finelabels_results.csv", index=False)

print("\nGSE114725 fine-label LIANA complete")

GSE114725: 44662 cells
GSE176078: 91425 cells
  Excluding from GSE114725_TUMOR (< 50 cells): ['Non-T-cell contamination (from T cells parent cluster)']
  Running LIANA on GSE114725_TUMOR: 19185 cells, 22 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt


  Excluding from GSE114725_NORMAL (< 50 cells): ['LAM-like macrophages', 'B cells', 'NKT cells', 'Mixed/stromal-contaminated (CD8+fibroblast signal)', 'Non-classical monocytes (CD16+)', 'Mast cells', 'pDC', 'Cycling CD8 T cells', 'Non-T-cell contamination (from T cells parent cluster)', 'Lipid-laden/Foam-cell macrophages']
  Running LIANA on GSE114725_NORMAL: 3974 cells, 13 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_scale.py:199: RuntimeWarning: invalid value encountered in sqrt
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packag


GSE114725 fine-label LIANA complete


In [4]:
tumor_results = all_liana_results_fine["GSE114725_TUMOR"]
normal_results = all_liana_results_fine["GSE114725_NORMAL"]

print("Tumour results — any NaN in key columns?")
print(tumor_results[["magnitude_rank", "specificity_rank"]].isna().sum())
print(f"Total rows: {len(tumor_results)}")

print("\nNormal results — any NaN in key columns?")
print(normal_results[["magnitude_rank", "specificity_rank"]].isna().sum())
print(f"Total rows: {len(normal_results)}")

Tumour results — any NaN in key columns?
magnitude_rank      62436
specificity_rank        0
dtype: int64
Total rows: 62436

Normal results — any NaN in key columns?
magnitude_rank      21801
specificity_rank    21801
dtype: int64
Total rows: 21801


In [5]:
# Check whether the same NaN pattern shows up specifically for one or
# a few cell types, or if it's genuinely universal across all rows
print("Tumour — NaN magnitude_rank by target/source cell type:")
print(tumor_results[tumor_results["magnitude_rank"].isna()]["source"].value_counts().head(10))

# Check the actual scaled data for zero-variance genes within any
# fine-grained group, in the Tumour subset specifically
adata1_tumor_check = adata1[adata1.obs["tissue"] == "TUMOR"].copy()
adata1_tumor_check = adata1_tumor_check[adata1_tumor_check.obs["cell_type_fine"].notna()]
for ct in adata1_tumor_check.obs["cell_type_fine"].unique():
    sub = adata1_tumor_check[adata1_tumor_check.obs["cell_type_fine"] == ct]
    if sub.n_obs < 60:  # focus on smaller groups first
        X = sub.raw.X if sub.raw is not None else sub.X
        if hasattr(X, "toarray"): X = X.toarray()
        n_zero_var_genes = (X.var(axis=0) == 0).sum()
        print(f"  {ct} (n={sub.n_obs}): {n_zero_var_genes} zero-variance genes")

Tumour — NaN magnitude_rank by target/source cell type:
source
Activated CD8 T cells                2838
Antigen-presenting macrophages       2838
B cells                              2838
CD4 Activated T cells                2838
CD4 Naive/Resting T cells            2838
Complement-high macrophages          2838
Cycling CD8 T cells                  2838
Effector CD8 T cells                 2838
LAM-like macrophages                 2838
Lipid-laden/Foam-cell macrophages    2838
Name: count, dtype: int64
  Non-T-cell contamination (from T cells parent cluster) (n=30): 6516 zero-variance genes


In [7]:
import os
for f in os.listdir(PROJECT_DIR / "results" / "phase3_liana"):
    print(f)

GSE114725_liana_NORMAL.csv
GSE114725_liana_NORMAL_top10.csv
GSE114725_liana_TUMOR.csv
GSE114725_liana_TUMOR_top10.csv
GSE176078_liana_ERplus.csv
GSE176078_liana_ERplus_top10.csv
GSE176078_liana_HER2plus.csv
GSE176078_liana_HER2plus_top10.csv
GSE176078_liana_TNBC.csv
GSE176078_liana_TNBC_top10.csv


In [8]:
original_tumor = pd.read_csv(PROJECT_DIR / "results" / "phase3_liana" / "GSE114725_liana_TUMOR.csv")
print("Original parent-level LIANA — NaN check:")
print(original_tumor[["magnitude_rank", "specificity_rank"]].isna().sum())
print(f"Total rows: {len(original_tumor)}")

Original parent-level LIANA — NaN check:
magnitude_rank      0
specificity_rank    0
dtype: int64
Total rows: 6478


In [9]:
import liana
print(f"LIANA version: {liana.__version__}")

LIANA version: 1.7.3


In [10]:
print("Non-T-cell contamination category still present in Tumour run?")
print("Non-T-cell contamination (from T cells parent cluster)" in tumor_results["source"].unique())

Non-T-cell contamination category still present in Tumour run?
False


In [11]:
# ----------------------------
# Bisection test — run LIANA on ONLY the largest fine categories
# (well above the 50-cell threshold), to isolate whether the NaN issue
# is related to group size/count, or something else entirely.
# ----------------------------
large_categories = adata1[adata1.obs["tissue"] == "TUMOR"].obs["cell_type_fine"].value_counts()
large_categories = large_categories[large_categories >= 500].index.tolist()
print(f"Testing with only these large categories: {large_categories}")

mask_large = (adata1.obs["tissue"] == "TUMOR") & (adata1.obs["cell_type_fine"].isin(large_categories))
test_results = run_liana_fine(adata1, mask_large.values, "GSE114725_TUMOR_bisection_test")

print("\nBisection test — NaN check:")
print(test_results[["magnitude_rank", "specificity_rank"]].isna().sum())
print(f"Total rows: {len(test_results)}")

Testing with only these large categories: ['Activated CD8 T cells', 'CD4 Activated T cells', 'Mixed/stromal-contaminated (CD8+fibroblast signal)', 'Monocyte-like macrophages', 'Complement-high macrophages', 'LAM-like macrophages', 'Regulatory T cells (Tregs, CD4+)', 'Antigen-presenting macrophages', 'B cells', 'Lipid-laden/Foam-cell macrophages', 'True NK cells', 'Effector CD8 T cells', 'NK-like CD8 T cells']
  Running LIANA on GSE114725_TUMOR_bisection_test: 16675 cells, 13 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt



Bisection test — NaN check:
magnitude_rank      21801
specificity_rank        0
dtype: int64
Total rows: 21801


In [12]:
adata1_tumor_check2 = adata1[adata1.obs["tissue"] == "TUMOR"].copy()
adata1_tumor_check2 = adata1_tumor_check2[adata1_tumor_check2.obs["cell_type_fine"].notna()]
adata1_tumor_check2 = adata1_tumor_check2[
    adata1_tumor_check2.obs["cell_type_fine"].isin(large_categories)
].copy()

X = adata1_tumor_check2.raw.X if adata1_tumor_check2.raw is not None else adata1_tumor_check2.X
if hasattr(X, "toarray"): X = X.toarray()

variances = X.var(axis=0)
print(f"Genes with exactly zero variance: {(variances == 0).sum()}")
print(f"Genes with tiny negative variance (floating point issue): {(variances < 0).sum()}")
print(f"Minimum variance value: {variances.min()}")
print(f"Any NaN in the raw expression matrix itself: {np.isnan(X).sum()}")
print(f"Any Inf in the raw expression matrix itself: {np.isinf(X).sum()}")

Genes with exactly zero variance: 4
Genes with tiny negative variance (floating point issue): 0
Minimum variance value: 0.0
Any NaN in the raw expression matrix itself: 0
Any Inf in the raw expression matrix itself: 0


In [14]:
zero_var_genes = adata1_tumor_check2.raw.var_names[variances == 0].tolist()
print(f"Zero-variance genes: {zero_var_genes}")

Zero-variance genes: ['HS6ST2', 'SLC22A3', 'TENM2', 'TMSB15A']


In [15]:
# ----------------------------
# Updated LIANA runner — filters zero-variance genes per subset before
# running rank_aggregate. Root cause: with fine-grained categories,
# individual cell types can have genes with zero variance (uniformly
# zero or uniformly equal expression), which produces division-by-zero
# during z-scoring in LIANA's magnitude-based sub-methods, propagating
# NaN through the entire rank_aggregate output for magnitude_rank.
# ----------------------------
def run_liana_fine_v2(adata, subset_mask, label, groupby_col="cell_type_fine", min_cells=MIN_CELLS_FOR_LIANA):
    adata_sub = adata[subset_mask].copy()
    adata_sub = adata_sub[adata_sub.obs[groupby_col].notna()].copy()
    adata_sub = adata_sub[~adata_sub.obs[groupby_col].isin(EXCLUDE_FROM_LIANA)].copy()

    cell_counts = adata_sub.obs[groupby_col].value_counts()
    viable_types = cell_counts[cell_counts >= min_cells].index.tolist()
    excluded_types = cell_counts[cell_counts < min_cells].index.tolist()
    if excluded_types:
        print(f"  Excluding from {label} (< {min_cells} cells): {excluded_types}")

    adata_sub = adata_sub[adata_sub.obs[groupby_col].isin(viable_types)].copy()
    adata_sub.obs[groupby_col] = adata_sub.obs[groupby_col].astype(str)

    # NEW: filter zero-variance genes from the RAW layer before LIANA runs
    X_check = adata_sub.raw.X if adata_sub.raw is not None else adata_sub.X
    if hasattr(X_check, "toarray"):
        X_check = X_check.toarray()
    variances = X_check.var(axis=0)
    n_zero_var = (variances == 0).sum()
    if n_zero_var > 0:
        print(f"  Removing {n_zero_var} zero-variance genes before LIANA (subset: {label})")
        keep_genes = adata_sub.raw.var_names[variances != 0]
        adata_sub_raw_filtered = adata_sub.raw.to_adata()[:, keep_genes].copy()
        adata_sub.raw = adata_sub_raw_filtered
        # Also filter the main matrix to the same gene set where present
        common_genes = [g for g in adata_sub.var_names if g in keep_genes]
        adata_sub = adata_sub[:, common_genes].copy()

    print(f"  Running LIANA on {label}: {adata_sub.n_obs} cells, {adata_sub.obs[groupby_col].nunique()} cell types")

    li.mt.rank_aggregate(
        adata_sub,
        groupby=groupby_col,
        expr_prop=0.1,
        verbose=False,
        use_raw=False,
    )
    results = adata_sub.uns["liana_res"].copy()
    del adata_sub
    gc.collect()
    return results

print("LIANA runner v2 ready (zero-variance gene filtering)")

LIANA runner v2 ready (zero-variance gene filtering)


In [16]:
mask_tumor = adata1.obs["tissue"] == "TUMOR"
test_results_v2 = run_liana_fine_v2(adata1, mask_tumor.values, "GSE114725_TUMOR_v2_test")

print("\nv2 test — NaN check:")
print(test_results_v2[["magnitude_rank", "specificity_rank"]].isna().sum())
print(f"Total rows: {len(test_results_v2)}")

  Excluding from GSE114725_TUMOR_v2_test (< 50 cells): ['Non-T-cell contamination (from T cells parent cluster)']
  Removing 1 zero-variance genes before LIANA (subset: GSE114725_TUMOR_v2_test)
  Running LIANA on GSE114725_TUMOR_v2_test: 19185 cells, 22 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in sqrt



v2 test — NaN check:
magnitude_rank      62436
specificity_rank        0
dtype: int64
Total rows: 62436


In [17]:
# ----------------------------
# Check zero-variance genes WITHIN each individual cell type group,
# not just globally across the combined population
# ----------------------------
adata1_tumor_full = adata1[adata1.obs["tissue"] == "TUMOR"].copy()
adata1_tumor_full = adata1_tumor_full[adata1_tumor_full.obs["cell_type_fine"].notna()]

problem_genes = set()
for ct in adata1_tumor_full.obs["cell_type_fine"].unique():
    sub = adata1_tumor_full[adata1_tumor_full.obs["cell_type_fine"] == ct]
    if sub.n_obs < 50:
        continue
    X = sub.raw.X
    if hasattr(X, "toarray"): X = X.toarray()
    variances = X.var(axis=0)
    n_zero = (variances == 0).sum()
    if n_zero > 0:
        genes = sub.raw.var_names[variances == 0].tolist()
        problem_genes.update(genes)
        print(f"  {ct} (n={sub.n_obs}): {n_zero} zero-variance genes")

print(f"\nTotal unique problem genes across all groups: {len(problem_genes)}")

  CD4 Activated T cells (n=3455): 982 zero-variance genes
  Regulatory T cells (Tregs, CD4+) (n=819): 2281 zero-variance genes
  True NK cells (n=680): 2697 zero-variance genes
  Activated CD8 T cells (n=4149): 812 zero-variance genes
  B cells (n=763): 2702 zero-variance genes
  Effector CD8 T cells (n=636): 2825 zero-variance genes
  LAM-like macrophages (n=898): 2131 zero-variance genes
  Antigen-presenting macrophages (n=763): 1442 zero-variance genes
  Resting/Resident macrophages (n=397): 2279 zero-variance genes
  Complement-high macrophages (n=921): 887 zero-variance genes
  NK-like CD8 T cells (n=523): 2731 zero-variance genes
  Monocytes/DC (n=280): 4554 zero-variance genes
  Monocyte-like macrophages (n=1030): 1051 zero-variance genes
  Mast cells (n=471): 851 zero-variance genes
  Lipid-laden/Foam-cell macrophages (n=732): 1228 zero-variance genes
  Mixed/stromal-contaminated (CD8+fibroblast signal) (n=1306): 391 zero-variance genes
  CD4 Naive/Resting T cells (n=201): 5561

In [1]:
# Document the limitation directly
print("magnitude_rank could not be reliably computed at cell_type_fine resolution:")
print(f"7,974 of 14,800 genes (54%) show zero variance within at least one of the")
print(f"22 fine-grained cell type groups, causing z-scoring failure in LIANA's")
print(f"magnitude-based sub-methods.")
print(f"NOTE: specificity_rank was fully computable for this specific condition")
print(f"(GSE114725 Tumour), but later testing (Normal, GSE176078 ER+) showed")
print(f"specificity_rank ALSO fails in those conditions — this does not generalise.")
print(f"Overall conclusion: fine-resolution LIANA is unreliable across conditions,")
print(f"not specifically isolated to magnitude_rank. See final summary.")

magnitude_rank could not be reliably computed at cell_type_fine resolution:
7,974 of 14,800 genes (54%) show zero variance within at least one of the
22 fine-grained cell type groups, causing z-scoring failure in LIANA's
magnitude-based sub-methods.
NOTE: specificity_rank was fully computable for this specific condition
(GSE114725 Tumour), but later testing (Normal, GSE176078 ER+) showed
specificity_rank ALSO fails in those conditions — this does not generalise.
Overall conclusion: fine-resolution LIANA is unreliable across conditions,
not specifically isolated to magnitude_rank. See final summary.


In [19]:
mask_normal = adata1.obs["tissue"] == "NORMAL"
normal_results = run_liana_fine(adata1, mask_normal.values, "GSE114725_NORMAL")

print("\nNormal — NaN check (expect magnitude_rank NaN, specificity_rank clean):")
print(normal_results[["magnitude_rank", "specificity_rank"]].isna().sum())

all_liana_results_fine["GSE114725_TUMOR"] = tumor_results  # from earlier
all_liana_results_fine["GSE114725_NORMAL"] = normal_results
normal_results.to_csv(RESULTS_DIR / "GSE114725_NORMAL_liana_finelabels_results.csv", index=False)

print("\nGSE114725 fine-label LIANA complete (both conditions)")

  Excluding from GSE114725_NORMAL (< 50 cells): ['LAM-like macrophages', 'B cells', 'NKT cells', 'Mixed/stromal-contaminated (CD8+fibroblast signal)', 'Non-classical monocytes (CD16+)', 'Mast cells', 'pDC', 'Cycling CD8 T cells', 'Non-T-cell contamination (from T cells parent cluster)', 'Lipid-laden/Foam-cell macrophages']
  Running LIANA on GSE114725_NORMAL: 3974 cells, 13 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_scale.py:199: RuntimeWarning: invalid value encountered in sqrt
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packag


Normal — NaN check (expect magnitude_rank NaN, specificity_rank clean):
magnitude_rank      21801
specificity_rank    21801
dtype: int64

GSE114725 fine-label LIANA complete (both conditions)


In [20]:
print(f"Total rows: {len(normal_results)}")
print(normal_results.head(10))
print("\nAny completely empty/duplicate rows?")
print(normal_results.isna().all(axis=1).sum())

Total rows: 21801
                  source                 target ligand_complex  \
0  Activated CD8 T cells  Activated CD8 T cells        ADCYAP1   
1  Activated CD8 T cells  Activated CD8 T cells        ADCYAP1   
2  Activated CD8 T cells  Activated CD8 T cells            ADM   
3  Activated CD8 T cells  Activated CD8 T cells            ADM   
4  Activated CD8 T cells  Activated CD8 T cells           APOE   
5  Activated CD8 T cells  Activated CD8 T cells           BTLA   
6  Activated CD8 T cells  Activated CD8 T cells           BTLA   
7  Activated CD8 T cells  Activated CD8 T cells             C3   
8  Activated CD8 T cells  Activated CD8 T cells             C3   
9  Activated CD8 T cells  Activated CD8 T cells          CCL13   

  receptor_complex  lr_means  cellphone_pvals  expr_prod  scaled_weight  \
0            ADRB2 -0.081369            0.969   0.006185      -0.038688   
1            VIPR2 -0.045928            0.610   0.001897      -0.023801   
2            ADRB2 -0.114333  

In [3]:
import gc
import numpy as np
import pandas as pd
import scanpy as sc
import liana as li
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
RESULTS_DIR = PROJECT_DIR / "results" / "phase4_finelabels_liana"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase4_finelabels_liana"

EXCLUDE_FROM_LIANA = ["Unassigned (n=28, doublet/mixed-identity artefact)",
                       "Unassigned (n=91, stromal/RBC contamination artefact)"]
MIN_CELLS_FOR_LIANA = 50

def run_liana_fine(adata, subset_mask, label, groupby_col="cell_type_fine", min_cells=MIN_CELLS_FOR_LIANA):
    adata_sub = adata[subset_mask].copy()
    adata_sub = adata_sub[adata_sub.obs[groupby_col].notna()].copy()
    adata_sub = adata_sub[~adata_sub.obs[groupby_col].isin(EXCLUDE_FROM_LIANA)].copy()

    cell_counts = adata_sub.obs[groupby_col].value_counts()
    viable_types = cell_counts[cell_counts >= min_cells].index.tolist()
    excluded_types = cell_counts[cell_counts < min_cells].index.tolist()
    if excluded_types:
        print(f"  Excluding from {label} (< {min_cells} cells): {excluded_types}")

    adata_sub = adata_sub[adata_sub.obs[groupby_col].isin(viable_types)].copy()
    adata_sub.obs[groupby_col] = adata_sub.obs[groupby_col].astype(str)
    print(f"  Running LIANA on {label}: {adata_sub.n_obs} cells, {adata_sub.obs[groupby_col].nunique()} cell types")

    li.mt.rank_aggregate(
        adata_sub,
        groupby=groupby_col,
        expr_prop=0.1,
        verbose=False,
        use_raw=False,
    )
    results = adata_sub.uns["liana_res"].copy()
    del adata_sub
    gc.collect()
    return results

print("Setup complete, ready to load adata2")

Setup complete, ready to load adata2


In [4]:
adata2 = sc.read_h5ad(PROCESSED_DIR / "GSE176078_phase2_v2_annotated_corrected_finelabels.h5ad")
print(f"Loaded: {adata2.n_obs} cells")

Loaded: 91425 cells


In [5]:
mask_er = adata2.obs["subtype"] == "ER+"
er_results = run_liana_fine(adata2, mask_er.values, "GSE176078_ERplus")
er_results.to_csv(RESULTS_DIR / "GSE176078_ERplus_liana_finelabels_results.csv", index=False)

print("\nER+ — NaN check:")
print(er_results[["magnitude_rank", "specificity_rank"]].isna().sum())
print(f"Total rows: {len(er_results)}")

del mask_er
gc.collect()

  Running LIANA on GSE176078_ERplus: 33552 cells, 24 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_scale.py:199: RuntimeWarning: invalid value encountered in sqrt
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packag


ER+ — NaN check:
magnitude_rank      180864
specificity_rank    180864
dtype: int64
Total rows: 180864


0

In [6]:
# ----------------------------
# Bisection test — split ER+'s 24 categories into two halves, test
# each independently, to distinguish "too many categories" from
# "one specific problematic category/pair" as the actual cause.
# ----------------------------
adata2_er = adata2[adata2.obs["subtype"] == "ER+"].copy()
adata2_er = adata2_er[adata2_er.obs["cell_type_fine"].notna()]
adata2_er = adata2_er[~adata2_er.obs["cell_type_fine"].isin(EXCLUDE_FROM_LIANA)]

all_categories = adata2_er.obs["cell_type_fine"].value_counts()
viable_categories = all_categories[all_categories >= MIN_CELLS_FOR_LIANA].index.tolist()
print(f"Total viable categories: {len(viable_categories)}")

half_1 = viable_categories[:len(viable_categories)//2]
half_2 = viable_categories[len(viable_categories)//2:]
print(f"\nHalf 1 ({len(half_1)} categories): {half_1}")
print(f"\nHalf 2 ({len(half_2)} categories): {half_2}")

mask_half1 = adata2.obs["cell_type_fine"].isin(half_1) & (adata2.obs["subtype"] == "ER+")
half1_results = run_liana_fine(adata2, mask_half1.values, "ER+_half1_test")
print("\nHalf 1 — NaN check:")
print(half1_results[["magnitude_rank", "specificity_rank"]].isna().sum())

del mask_half1
gc.collect()

Total viable categories: 24

Half 1 (12 categories): ['Luminal epithelial', 'Endothelial cells', 'PVL', 'CAFs', 'Memory T cells', 'Macrophages', 'T cells', 'Epithelial (ambiguous)', 'Basal epithelial', 'Stress-Response/Activated CD8 T cells', 'Resting/Memory-like CD8 T cells', 'GZMK+ CD8 T cells']

Half 2 (12 categories): ['Plasma cells', 'B cells', 'Effector CD8 T cells', 'Cycling epithelial', 'True NK cells', 'Cytotoxic CD8 T cells (reclassified)', 'NK cells', 'NKT cells', 'pDC', 'Cycling T cells', 'Interferon-Response CD8 T cells', 'Exhausted CD8 T cells']
  Running LIANA on ER+_half1_test: 31034 cells, 12 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_scale.py:199: RuntimeWarning: invalid value encountered in sqrt
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packag


Half 1 — NaN check:
magnitude_rank      45216
specificity_rank    45216
dtype: int64


0

In [7]:
mask_half2 = adata2.obs["cell_type_fine"].isin(half_2) & (adata2.obs["subtype"] == "ER+")
half2_results = run_liana_fine(adata2, mask_half2.values, "ER+_half2_test")
print("\nHalf 2 — NaN check:")
print(half2_results[["magnitude_rank", "specificity_rank"]].isna().sum())

del mask_half2
gc.collect()

  Running LIANA on ER+_half2_test: 2518 cells, 12 cell types


C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\anndata\_core\anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\_pipe_utils\_pre.py:168: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\scanpy\preprocessing\_scale.py:199: RuntimeWarning: invalid value encountered in sqrt
C:\Users\annam\anaconda3\envs\scrna\lib\functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packages\liana\method\sc\_liana_pipe.py:288: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
C:\Users\annam\anaconda3\envs\scrna\lib\site-packag


Half 2 — NaN check:
magnitude_rank      45216
specificity_rank    45216
dtype: int64


0